# Setting up imports

In [1]:
import os 
import torch 
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms,utils
from PIL import Image
import matplotlib.pyplot as plt
from torchmetrics.image import StructuralSimilarityIndexMeasure, PeakSignalNoiseRatio
import numpy as np 
import zipfile
import gradio as gr
from tqdm.auto import tqdm


# Dataset Preparation

In [2]:
# import os
# for root, dirs, files in os.walk('/kaggle/input'):
#     if 'train' in root and 'anime' in root:
#         print(f"COPY THIS PATH: {root}")
#         break  


In [3]:
class Pix2PixDataset(Dataset):

    def __init__(self,root_dir,mode='anime',transform=None):
        self.root_dir=root_dir
        self.mode=mode
        self.transform=transform
        self.img_files=sorted([f for f in os.listdir(root_dir)if f.endswith(('.jpg','.png'))])


    def __len__(self):
        return len(self.img_files)

    def __getitem__(self,idx):
        img_path=os.path.join(self.root_dir,self.img_files[idx])
        img=Image.open(img_path).convert("RGB")
        if self.mode=='anime':
            w,h=img.size
            target = img.crop((0, 0, w // 2, h))
            sketch = img.crop((w // 2, 0, w, h))
           
        else:
            sketch=img
            target=img

        if self.transform:
            sketch = self.transform(sketch)
            target = self.transform(target)
        
        return sketch,target

transform=transforms.Compose([
transforms.Resize((256,256)),
transforms.ToTensor(),
transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
])


# Build our model archtiecture

In [4]:
# This is our FF network
class CNNBlock(nn.Module):
    def __init__(self,in_channels,out_channels,stride):
        super().__init__()
        self.conv=nn.Sequential(
            nn.Conv2d(in_channels,out_channels,4,stride,1,bias=False,padding_mode="reflect"),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2),
        )
    def forward(self,x):
        return self.conv(x)

class Discriminator(nn.Module):
    def __init__(self,in_channels=3,features=[64,128,256,512]):
        super().__init__()
        self.initial=nn.Sequential(
            nn.Conv2d(in_channels*2,features[0],kernel_size=4,stride=2,padding=1),
            nn.LeakyReLU(0.2)
            
        )
        layers=[]
        in_c=features[0]
        for feature in features:
            layers.append(CNNBlock(in_c,feature,stride=1 if feature==features[-1] else 2))
            in_c=feature
        layers.append(nn.Conv2d(in_c,feature,stride=1,kernel_size=4,padding=1,padding_mode="reflect"))
        self.model=nn.Sequential(*layers)

    def forward(self,x,y):
        return self.model(self.initial(torch.cat([x,y],dim=1)))


class UNetBlock(nn.Module):
    def __init__(self,in_channels,out_channels,down=True,act="relu",use_dropout=False):
        super().__init__()
        self.conv=nn.Sequential(
            nn.Conv2d(in_channels,out_channels,4,2,1,bias=False,padding_mode="reflect") if down
            else nn.ConvTranspose2d(in_channels,out_channels,4,2,1,bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU() if act=="relu" else nn.LeakyReLU(0.2),
        )
    
        self.use_dropout=use_dropout
        self.dropout=nn.Dropout(0.5)
    def forward(self,x):
        x=self.conv(x)
        return self.dropout(x) if self.use_dropout else x


class Generator(nn.Module):
    def __init__(self,in_channels=3,features=64):
        super().__init__()
        self.initial_down=nn.Sequential(
            nn.Conv2d(in_channels,features,4,2,1,padding_mode="reflect"),
            nn.LeakyReLU(0.2),
            
        )
        self.down1=UNetBlock(features,features*2,down=True,act="leaky",use_dropout=False)
        self.down2=UNetBlock(features*2,features*4,down=True,act="leaky",use_dropout=False)
        self.down3=UNetBlock(features*4,features*8,down=True,act="leaky",use_dropout=False)
        self.down4=UNetBlock(features*8,features*8,down=True,act="leaky",use_dropout=False)
        self.down5=UNetBlock(features*8,features*8,down=True,act="leaky",use_dropout=False)
        self.down6=UNetBlock(features*8,features*8,down=True,act="leaky",use_dropout=False)
    
        self.bottleneck=nn.Sequential(
            nn.Conv2d(features*8,features*8,4,2,1,padding_mode='reflect'),
            nn.ReLU()
            
        )
        self.up1=UNetBlock(features*8,features*8,down=False,act="relu",use_dropout=True)
        self.up2=UNetBlock(features*16,features*8,down=False,act="relu",use_dropout=True)
        self.up3=UNetBlock(features*16,features*8,down=False,act="relu",use_dropout=True)
        self.up4=UNetBlock(features*16,features*8,down=False,act="relu",use_dropout=True)
        self.up5=UNetBlock(features*16,features*4,down=False,act="relu",use_dropout=False)
        self.up6=UNetBlock(features*8,features*2,down=False,act="relu",use_dropout=False)
        self.up7=UNetBlock(features*4,features,down=False,act="relu",use_dropout=False)
    
        self.final_up=nn.Sequential(
            nn.ConvTranspose2d(features*2,in_channels,kernel_size=4,stride=2,padding=1),
            nn.Tanh(),
        )

    def forward(self,x):
        d1=self.initial_down(x)
        d2=self.down1(d1)
        d3=self.down2(d2)
        d4=self.down3(d3)
        d5=self.down4(d4)
        d6=self.down5(d5)
        d7=self.down6(d6)
        bn=self.bottleneck(d7)
        u1=self.up1(bn)
        u2=self.up2(torch.cat([u1, d7], dim=1))
        u3=self.up3(torch.cat([u2, d6], dim=1))
        u4=self.up4(torch.cat([u3, d5], dim=1))
        u5=self.up5(torch.cat([u4, d4], dim=1))
        u6=self.up6(torch.cat([u5, d3], dim=1))
        u7=self.up7(torch.cat([u6, d2], dim=1))
        return self.final_up(torch.cat([u7,d1], dim=1))
        


# Training loop

In [ ]:
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
LEARNING_RATE=2e-4
BATCH_SIZE=32
LAMBDA_L1=100
NUM_EPOCHS=50

gen = Generator().to(DEVICE)
disc = Discriminator().to(DEVICE)


if torch.cuda.device_count() > 1:
    gen = nn.DataParallel(gen)
    disc = nn.DataParallel(disc)

opt_disc = optim.Adam(disc.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
opt_gen = optim.Adam(gen.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
BCE = nn.BCEWithLogitsLoss()
L1_LOSS = nn.L1Loss()
scaler = torch.amp.GradScaler('cuda')

def train_fn(loader, disc, gen, opt_disc, opt_gen, L1_LOSS, BCE, scaler):
    loop = tqdm(loader, leave=True)
    for x, y in loop:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.cuda.amp.autocast():
            y_fake = gen(x)
            D_real = disc(x, y)
            D_fake = disc(x, y_fake.detach())
            loss_D = (BCE(D_real, torch.ones_like(D_real)) + BCE(D_fake, torch.zeros_like(D_fake))) / 2
        opt_disc.zero_grad()
        scaler.scale(loss_D).backward()
        scaler.step(opt_disc)
        with torch.cuda.amp.autocast():
            D_fake = disc(x, y_fake)
            loss_G = BCE(D_fake, torch.ones_like(D_fake)) + LAMBDA_L1 * L1_LOSS(y_fake, y)
        opt_gen.zero_grad()
        scaler.scale(loss_G).backward()
        scaler.step(opt_gen)
        scaler.update()
        loop.set_postfix(D_loss=loss_D.item(), G_loss=loss_G.item())
        
PATH = "/kaggle/input/datasets/ktaebum/anime-sketch-colorization-pair/data/data/train"
dataset = Pix2PixDataset(root_dir=PATH, mode='anime', transform=transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
for epoch in range(NUM_EPOCHS):
    train_fn(loader, disc, gen, opt_disc, opt_gen, L1_LOSS, BCE, scaler)
    torch.save(gen.state_dict(), "gen_latest.pth")
    torch.save(disc.state_dict(), "disc_latest.pth")
    with zipfile.ZipFile("latest_checkpoint.zip", 'w') as zf:
        zf.write("gen_latest.pth")
        zf.write("disc_latest.pth")

  0%|          | 0/445 [00:00<?, ?it/s]

/tmp/ipykernel_55/75298808.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_55/75298808.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  0%|          | 0/445 [00:00<?, ?it/s]

  0%|          | 0/445 [00:00<?, ?it/s]

  0%|          | 0/445 [00:00<?, ?it/s]

...

  0%|          | 0/445 [00:00<?, ?it/s]

  0%|          | 0/445 [00:00<?, ?it/s]

  0%|          | 0/445 [00:00<?, ?it/s]

  0%|          | 0/445 [00:00<?, ?it/s]

  0%|          | 0/445 [00:00<?, ?it/s]

# Evaluation

In [ ]:
ssim = StructuralSimilarityIndexMeasure().to(DEVICE); psnr = PeakSignalToNoiseRatio().to(DEVICE)
def evaluate(loader, gen):
    gen.eval(); avg_ssim, avg_psnr = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE); fake = gen(x)
            avg_ssim.append(ssim(fake, y).item()); avg_psnr.append(psnr(fake, y).item())
    print(f"Avg SSIM: {np.mean(avg_ssim):.4f} | Avg PSNR: {np.mean(avg_psnr):.2f}")
    gen.train()
evaluate(loader, gen)

# Gradio App

In [ ]:
def predict(input_img):
    gen.eval(); input_tensor = transform(input_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prediction = (gen(input_tensor) * 0.5 + 0.5).squeeze().cpu()
    return transforms.ToPILImage()(prediction)

gr.Interface(fn=predict, inputs=gr.Image(type="pil"), outputs=gr.Image(type="pil"), title="Pix2Pix Real-Time Inference").launch(share=True)